<a href="https://colab.research.google.com/github/evmei26/Fantasy-Football-Project/blob/main/FantasyNLPBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gc
import torch

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
import pandas as pd
import xgboost as xgb
import numpy as np


from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction import text
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from xgboost import plot_importance, plot_tree
from huggingface_hub import login
from sklearn.utils.class_weight import compute_sample_weight
from google.colab import userdata
# RoBERTa using brainrot twitter data

#STEP 1: Choose model
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


hf_token = userdata.get('HuggingFace')
login(token=hf_token)


# STEP 2: Use dumbed down GoEmotions: Three categories (Positive, Neutral, Negative)
# There are only two options: draft or no draft, so I don't need to train the model to separate into almost 30 emotions
goemo = load_dataset("google-research-datasets/go_emotions", "simplified")

# GoEmotions simplified already collapses to positive/negative/neutral
print(goemo["train"].features)


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)


#STEP 3: Extract the emotions and bucket them into 3 categories
emotion_names = goemo["train"].features["labels"].feature.names

#Key: 0: Negative, 1: Neutral, 2: Positive
emotion_to_simplified_label = {
    # Positive
    'admiration': 2, 'amusement': 2, 'approval': 2, 'caring': 2, 'desire': 2,
    'excitement': 2, 'gratitude': 2, 'joy': 2, 'love': 2, 'optimism': 2,
    'pride': 2, 'relief': 2, 'surprise': 2,
    # Negative
    'anger': 0, 'annoyance': 0, 'disappointment': 0, 'disapproval': 0,
    'disgust': 0, 'embarrassment': 0, 'fear': 0, 'grief': 0, 'nervousness': 0,
    'remorse': 0, 'sadness': 0,
    # Neutral
    'confusion': 1, 'curiosity': 1, 'realization': 1, 'neutral': 1
}

#STEP 4: Turning words into numbers so the clanker can understand
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    # Remap labels for each example in the batch
    remapped_labels = []
    for original_label_list in batch["labels"]:
        # go_emotions posts can have multiple labels, so we take the dominant one and ignore the others
        original_label_idx = original_label_list[0]
        emotion_name = emotion_names[original_label_idx]

        if emotion_name in emotion_to_simplified_label:
            remapped_labels.append(emotion_to_simplified_label[emotion_name])
        else:
            # Just in case I missed an emotion or two
            raise ValueError(f"Emotion '{emotion_name}' (index {original_label_idx}) not found in simplified mapping.")

    tokens["labels"] = remapped_labels
    return tokens

tokenized = goemo.map(tokenize, batched=True)


In [ ]:

#logits are the raw prediction, still in clanker speak

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./roberta-sentiment",
    num_train_epochs=3,          # how many times it sees full dataset
    per_device_train_batch_size=16, #how many simultaneous points it processes during training
    per_device_eval_batch_size=32, #Same as above but for validation
    learning_rate=2e-5, #How dramatic it changes after each mistake. Low since it's a pretrained model
    warmup_steps=500,  #Starts very low for this many steps, then approaches learning rate. Protects pretrained knowledge
    weight_decay=0.01,           # prevents overfitting
    eval_strategy="epoch", #Running tests after an "epoch" three times, making checkpoints each time
    save_strategy="epoch",
    load_best_model_at_end=True, #use the best model (obviously)
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics
)



trainer.train()

I'm doing this so google doesn't stop my code pls don't stop my model I need this
jjjjjj

huiguyftfygiyfujhhgfvvbnhh



I'm still here
still here

In [ ]:
#PULL REDDIT POSTS

import praw
import pandas as pd
import datetime as dt


